# Complex Vectors, the Hermitian Inner Product, and Norms

Vectors with complex entries live in $\mathbb{C}^n$. Adding them and scaling them works exactly as in $\mathbb{R}^n$, but measuring them does not: the familiar dot product $\sum u_j v_j$ gives the nonzero vector $(1, i)$ "length squared" $1 + i^2 = 0$. This notebook builds the right tool, the **Hermitian inner product** $\langle u, v\rangle = \sum \overline{u_j}\, v_j$, and shows that with one conjugation in place, lengths, orthogonality, the Cauchy–Schwarz inequality and Gram–Schmidt all carry over to $\mathbb{C}^n$.

This is notebook 03 of module 04 (**Complex linear algebra**) in the **Learning Complex Numbers** series.

## 1. Introduction

Complex vectors are everywhere that phases matter: a quantum state is a unit vector in $\mathbb{C}^n$, a sampled signal's Fourier coefficients form a vector in $\mathbb{C}^n$, and the eigenvectors of the rotation matrices of notebook 02 were complex vectors. To talk about how *long* such a vector is, or whether two of them are *perpendicular*, we need an inner product.

The guiding fact is from module 00: for a single complex number, $|z|^2 = \bar z z$, not $z^2$. The square $z^2$ is complex and can be zero or negative; $\bar z z$ is always a nonnegative real number. The Hermitian inner product is exactly this idea applied entry by entry.

**Prerequisites**

- Conjugate and modulus, $|z|^2 = z\bar z$ — [Conjugate, modulus and division](../00_foundations/03_conjugate_modulus_and_division.ipynb).
- Polar form and the argument of a complex number — [Polar form, modulus and argument](../01_polar_form_and_euler/01_polar_form_modulus_and_argument.ipynb).
- Complex eigenvectors of real matrices — [Complex eigenvalues of real matrices](02_complex_eigenvalues_of_real_matrices.ipynb).
- The real dot product, lengths and angles — [Vector dot product](../../../LearningLinearAlgebra/hyperskill/01_vector_and_vector_spaces/014_vector_dot_product.ipynb) (LearningLinearAlgebra).

After completing this notebook you will be able to:

- Compute Hermitian inner products and norms in $\mathbb{C}^n$ by hand and with `np.vdot`, and explain why the conjugation is necessary.
- State the properties of $\langle\cdot,\cdot\rangle$ (conjugate symmetry, linearity in the second slot, positivity) and which slot gets conjugated in each convention.
- Decide orthogonality of complex vectors and explain why it is stronger than real perpendicularity.
- Prove and use the Cauchy–Schwarz inequality, and run Gram–Schmidt correctly in $\mathbb{C}^n$.

## 2. Theory

### 2.1 The space $\mathbb{C}^n$

$\mathbb{C}^n$ is the set of columns $v = (v_1, \dots, v_n)^T$ with $v_j \in \mathbb{C}$. Addition is entrywise and scalars are now **complex**: $cv = (cv_1, \dots, cv_n)$ for $c \in \mathbb{C}$. It is an $n$-dimensional vector space over $\mathbb{C}$ (basis $e_1, \dots, e_n$), and, forgetting complex scalars, a $2n$-dimensional real space via

$$
v = x + iy \;\longleftrightarrow\; \begin{pmatrix} x \\ y \end{pmatrix} \in \mathbb{R}^{2n}, \qquad x = \operatorname{Re}v,\ y = \operatorname{Im}v.
$$

### 2.2 Why the plain dot product fails

Try $u \cdot v = \sum_j u_j v_j$ on $z = (1, i)$:

$$
z \cdot z = 1^2 + i^2 = 0.
$$

A nonzero vector of "length" zero. Worse, $(i, 0)\cdot(i, 0) = -1$: negative length squared. The fix is the one-dimensional identity $|z|^2 = \bar z z$:

$$
\sum_j \overline{z_j}\, z_j = \sum_j |z_j|^2 = 1 + 1 = 2 \quad\text{for } z = (1, i).
$$

### 2.3 The Hermitian inner product

$$
\boxed{\;\langle u, v\rangle = \sum_{j=1}^n \overline{u_j}\, v_j = u^H v\;}
$$

where $u^H = \bar u^T$ is the conjugate transpose (notebook 04). Its properties, all immediate from the definition:

| Property | Statement |
|---|---|
| Conjugate symmetry | $\langle v, u\rangle = \overline{\langle u, v\rangle}$ |
| Linear in the second slot | $\langle u, av + bw\rangle = a\langle u, v\rangle + b\langle u, w\rangle$ |
| Conjugate-linear in the first slot | $\langle au + bw, v\rangle = \bar a\langle u, v\rangle + \bar b\langle w, v\rangle$ |
| Positive definite | $\langle v, v\rangle = \sum\lvert v_j\rvert^2 \ge 0$, with equality only for $v = 0$ |

**Conventions.** Physics, numerical linear algebra and numpy conjugate the **first** argument (as above): `np.vdot(u, v)` computes $\sum \overline{u_j}v_j$. Many pure-mathematics texts conjugate the **second** argument instead, $\langle u, v\rangle_{\text{math}} = \sum u_j\overline{v_j} = \overline{\langle u, v\rangle}$. Both give the same norms and the same notion of orthogonality; they differ only in which slot pulls scalars out with a bar. This series uses the first-slot convention throughout. Note also that `np.dot`, `@` and `np.inner` do **not** conjugate anything.

### 2.4 Norm, and the link to $\mathbb{R}^{2n}$

$$
\lVert v\rVert = \sqrt{\langle v, v\rangle} = \sqrt{\textstyle\sum_j |v_j|^2}, \qquad \lVert cv\rVert = |c|\,\lVert v\rVert \ (c \in \mathbb{C}).
$$

Since $|v_j|^2 = x_j^2 + y_j^2$, the complex norm equals the ordinary Euclidean norm of the real vector $(x, y) \in \mathbb{R}^{2n}$. More generally, writing $u = a + ib$ and $v = x + iy$:

$$
\langle u, v\rangle = (a \cdot x + b \cdot y) + i\,(a \cdot y - b \cdot x),
$$

so $\operatorname{Re}\langle u, v\rangle$ is the **real** dot product of the realified vectors. The imaginary part is new information that $\mathbb{R}^{2n}$ does not see.

**Geometry in $\mathbb{C}^1$.** For single numbers $w = |w|e^{i\alpha}$ and $z = |z|e^{i\beta}$:

$$
\langle w, z\rangle = \bar w z = |w||z|\,e^{i(\beta - \alpha)} = |w||z|\cos(\beta - \alpha) + i\,|w||z|\sin(\beta - \alpha).
$$

The real part is the dot product of the arrows $w$ and $z$ in the plane, the imaginary part their 2-D cross product, and the argument of $\langle w, z\rangle$ is the angle from $w$ to $z$.

### 2.5 Orthogonality

$u$ and $v$ are **orthogonal** if $\langle u, v\rangle = 0$. Because $\langle u, v\rangle = 0$ requires both real and imaginary parts to vanish, and $\operatorname{Im}\langle u, v\rangle = -\operatorname{Re}\langle u, iv\rangle$, complex orthogonality means:

$$
\langle u, v\rangle = 0 \iff u \perp v \text{ and } u \perp iv \text{ in } \mathbb{R}^{2n}.
$$

It is a stronger condition than real perpendicularity. For instance $z$ and $iz$ are perpendicular as real vectors, yet $\langle z, iz\rangle = i\lVert z\rVert^2 \ne 0$ — they span the same *complex* line. Orthogonality in $\mathbb{C}^n$ is orthogonality of complex lines.

**Pythagoras.** Expanding with the properties,

$$
\lVert u + v\rVert^2 = \lVert u\rVert^2 + \lVert v\rVert^2 + 2\operatorname{Re}\langle u, v\rangle,
$$

so if $\langle u, v\rangle = 0$ then $\lVert u + v\rVert^2 = \lVert u\rVert^2 + \lVert v\rVert^2$.

### 2.6 Projection and the Cauchy–Schwarz inequality

The projection of $u$ onto a nonzero $v$ is

$$
\operatorname{proj}_v u = \frac{\langle v, u\rangle}{\langle v, v\rangle}\, v.
$$

Mind the order: $v$ goes in the **first** (conjugated) slot. Then $u - \operatorname{proj}_v u$ is orthogonal to $v$:

$$
\langle v, u - \operatorname{proj}_v u\rangle = \langle v, u\rangle - \frac{\langle v, u\rangle}{\langle v, v\rangle}\langle v, v\rangle = 0,
$$

using linearity in the second slot. By Pythagoras, $\lVert u\rVert^2 = \lVert\operatorname{proj}_v u\rVert^2 + \lVert u - \operatorname{proj}_v u\rVert^2 \ge \lVert\operatorname{proj}_v u\rVert^2 = \frac{|\langle v, u\rangle|^2}{\lVert v\rVert^2}$. Rearranged:

$$
\boxed{\;|\langle u, v\rangle| \le \lVert u\rVert\,\lVert v\rVert\;}
$$

with equality exactly when $u - \operatorname{proj}_v u = 0$, i.e. when $u$ and $v$ are linearly dependent over $\mathbb{C}$. Consequently $\frac{\langle u, v\rangle}{\lVert u\rVert\lVert v\rVert}$ is a complex number in the closed **unit disk**, and the triangle inequality $\lVert u + v\rVert \le \lVert u\rVert + \lVert v\rVert$ follows as in $\mathbb{R}^n$.

### 2.7 Gram–Schmidt in $\mathbb{C}^n$

Given linearly independent $v_1, \dots, v_k$:

$$
w_m = v_m - \sum_{j < m} \langle e_j, v_m\rangle\, e_j, \qquad e_m = \frac{w_m}{\lVert w_m\rVert}.
$$

The result is **orthonormal**: $\langle e_j, e_m\rangle = \delta_{jm}$. The only change from $\mathbb{R}^n$ is that the coefficient must be $\langle e_j, v_m\rangle$, not $\langle v_m, e_j\rangle$; with the wrong order you subtract the *conjugate* coefficient and the result is not orthogonal unless that coefficient happens to be real. Orthonormal $e_1, \dots, e_n$ give the expansion

$$
v = \sum_j \langle e_j, v\rangle\, e_j, \qquad \lVert v\rVert^2 = \sum_j |\langle e_j, v\rangle|^2,
$$

the finite-dimensional Parseval identity that notebook 05 applies to the DFT.

## 3. Worked Examples

### Example 1 — Inner products and norms

$u = (1 + i,\ 2 - i)$, $v = (3,\ i)$.

$$
\langle u, v\rangle = \overline{(1 + i)}\cdot 3 + \overline{(2 - i)}\cdot i = (1 - i)3 + (2 + i)i = 3 - 3i + 2i - 1 = 2 - i.
$$

$\langle v, u\rangle = \overline{2 - i} = 2 + i$. Norms: $\lVert u\rVert^2 = |1 + i|^2 + |2 - i|^2 = 2 + 5 = 7$, $\lVert v\rVert^2 = 9 + 1 = 10$.

Cauchy–Schwarz: $|2 - i| = \sqrt5 \approx 2.24 \le \sqrt{70} \approx 8.37$. ✓

The unconjugated product gives $u\cdot v = 3(1 + i) + i(2 - i) = 4 + 5i$ — a different number with no geometric meaning.

### Example 2 — The vector that the dot product thinks is zero

$z = (1, i)$: $z\cdot z = 0$ but $\langle z, z\rangle = |1|^2 + |i|^2 = 2$, so $\lVert z\rVert = \sqrt2$, matching the realified vector $(1, 0, 0, 1) \in \mathbb{R}^4$.

### Example 3 — Which vectors are orthogonal?

$a = (1, i)$, $b = (1, -i)$, $c = (i, 1)$:

- $\langle a, b\rangle = 1\cdot 1 + \bar i\cdot(-i) = 1 + (-i)(-i) = 1 - 1 = 0$: orthogonal.
- $\langle a, c\rangle = 1\cdot i + (-i)\cdot 1 = 0$: orthogonal.
- $\langle b, c\rangle = 1\cdot i + \overline{(-i)}\cdot 1 = i + i = 2i \ne 0$: **not** orthogonal. In fact $c = ib$: same complex line.

(These $a, b$ are the rotation eigenvectors of notebook 02 — orthogonal, as notebook 04 will explain.)

### Example 4 — Gram–Schmidt in $\mathbb{C}^3$

$v_1 = (1, i, 0)$, $v_2 = (1, 1, i)$.

1. $\lVert v_1\rVert = \sqrt2$, $e_1 = \frac{1}{\sqrt2}(1, i, 0)$.
2. Coefficient $\langle e_1, v_2\rangle = \frac{1}{\sqrt2}(1\cdot 1 + \bar i\cdot 1 + 0) = \frac{1 - i}{\sqrt2}$.
3. $w_2 = v_2 - \frac{1 - i}{\sqrt2}e_1 = (1, 1, i) - \frac{1 - i}{2}(1, i, 0) = (1, 1, i) - \left(\frac{1 - i}{2}, \frac{1 + i}{2}, 0\right) = \left(\frac{1 + i}{2}, \frac{1 - i}{2}, i\right)$.
4. $\lVert w_2\rVert^2 = \frac12 + \frac12 + 1 = 2$, so $e_2 = \frac{1}{\sqrt2}\left(\frac{1 + i}{2}, \frac{1 - i}{2}, i\right)$.

Check: $\langle e_1, w_2\rangle = \frac{1}{\sqrt2}\left(\frac{1 + i}{2} + (-i)\frac{1 - i}{2}\right) = \frac{1}{\sqrt2}\cdot\frac{1 + i - i - 1}{2} = 0$. ✓

With the **wrong** coefficient $\langle v_2, e_1\rangle = \frac{1 + i}{\sqrt2}$ we would get $w_2' = (1, 1, i) - \frac{1 + i}{2}(1, i, 0) = \left(\frac{1 - i}{2}, \frac{3 - i}{2}, i\right)$, and $\langle e_1, w_2'\rangle = \frac{1}{\sqrt2}\left(\frac{1 - i}{2} - i\,\frac{3 - i}{2}\right) = \frac{1}{\sqrt2}\cdot\frac{-4i}{2} = -\sqrt2\, i \ne 0$.

## 4. Python Examples

### 4.1 `np.vdot` conjugates; `np.dot` and `@` do not

In [ ]:
import numpy as np

u = np.array([1 + 1j, 2 - 1j])
v = np.array([3, 1j])

print("np.vdot(u, v)        =", np.vdot(u, v), "  <- sum conj(u_j) v_j")
print("np.vdot(v, u)        =", np.vdot(v, u), "  <- the conjugate")
print("u.conj() @ v         =", u.conj() @ v)
print("np.dot(u, v) = u @ v =", np.dot(u, v), "  <- no conjugation")
print("np.inner(u, v)       =", np.inner(u, v), "  <- no conjugation either")
print("||u||^2 =", np.vdot(u, u).real, " ||v||^2 =", np.vdot(v, v).real)
print("np.linalg.norm(u)^2  =", np.linalg.norm(u) ** 2)

z = np.array([1, 1j])
print("\nz = (1, i):  z @ z =", z @ z, "   np.vdot(z, z) =", np.vdot(z, z))

> **Pitfall.** `np.vdot` flattens its arguments, so on 2-D arrays it returns a single number (the Frobenius inner product), not a matrix. For a matrix of inner products between columns use `A.conj().T @ B`.

### 4.2 Checking the properties symbolically

With sympy symbols for real and imaginary parts, conjugate symmetry and the two linearity rules are identities.

In [ ]:
import sympy as sp

symbols = sp.symbols("a1:5 b1:5 c1:5 d1:5 p q r s", real=True)
a, b, c, d = symbols[0:4], symbols[4:8], symbols[8:12], symbols[12:16]
p, q, r, s = symbols[16:]
u = sp.Matrix([a[0] + sp.I * b[0], a[1] + sp.I * b[1]])
v = sp.Matrix([c[0] + sp.I * d[0], c[1] + sp.I * d[1]])
alpha = p + sp.I * q


def inner(x, y):
    return sp.expand((x.H * y)[0, 0])


print("<v,u> - conj(<u,v>)          =", sp.simplify(inner(v, u) - sp.conjugate(inner(u, v))))
print("<u, alpha v> - alpha <u,v>   =", sp.simplify(inner(u, alpha * v) - alpha * inner(u, v)))
print("<alpha u, v> - conj(alpha)<u,v> =", sp.simplify(inner(alpha * u, v) - sp.conjugate(alpha) * inner(u, v)))
print("<u,u> =", sp.simplify(inner(u, u)), "  (a sum of squares)")

### 4.3 Real part = dot product in $\mathbb{R}^{2n}$

Realify both vectors by stacking real and imaginary parts and compare.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
u = rng.normal(size=4) + 1j * rng.normal(size=4)
v = rng.normal(size=4) + 1j * rng.normal(size=4)


def realify(z):
    return np.concatenate([z.real, z.imag])


print("<u, v>                    =", np.round(np.vdot(u, v), 6))
print("real dot of realified     =", np.round(realify(u) @ realify(v), 6))
print("same ||u|| both ways      :", np.isclose(np.linalg.norm(u), np.linalg.norm(realify(u))))
print("z and iz perpendicular in R^8:", np.isclose(realify(u) @ realify(1j * u), 0))
print("but <u, iu> =", np.round(np.vdot(u, 1j * u), 6), " = i ||u||^2 =", np.round(1j * np.vdot(u, u).real, 6))

### 4.4 Cauchy–Schwarz on random vectors

In [ ]:
import numpy as np

rng = np.random.default_rng(2)
worst_ratio = 0.0
for _ in range(10_000):
    u = rng.normal(size=5) + 1j * rng.normal(size=5)
    v = rng.normal(size=5) + 1j * rng.normal(size=5)
    ratio = abs(np.vdot(u, v)) / (np.linalg.norm(u) * np.linalg.norm(v))
    worst_ratio = max(worst_ratio, ratio)
print("largest |<u,v>| / (||u|| ||v||) over 10,000 random pairs:", round(worst_ratio, 4))

u = rng.normal(size=5) + 1j * rng.normal(size=5)
dependent = (2 - 3j) * u
print("equality for v = (2-3i) u:",
      np.isclose(abs(np.vdot(u, dependent)), np.linalg.norm(u) * np.linalg.norm(dependent)))

### 4.5 Gram–Schmidt, the right way and the wrong way

The function below takes a flag for the coefficient order so the failure is visible. The Gram matrix $E^H E$ of the output should be the identity.

In [ ]:
import numpy as np


def gram_schmidt(vectors, correct_order=True):
    orthonormal = []
    for v in vectors:
        w = v.astype(complex)
        for e in orthonormal:
            coefficient = np.vdot(e, v) if correct_order else np.vdot(v, e)
            w = w - coefficient * e
        orthonormal.append(w / np.linalg.norm(w))
    return np.column_stack(orthonormal)


v1 = np.array([1, 1j, 0])
v2 = np.array([1, 1, 1j])
v3 = np.array([0, 1j, 1 + 1j])

E = gram_schmidt([v1, v2, v3])
print("e2 * sqrt(2) =", np.round(E[:, 1] * np.sqrt(2), 6), " (Example 4)")
print("Gram matrix E^H E (correct):\n", np.round(E.conj().T @ E, 10) + (0 + 0j))   # + 0 tidies -0 signs

E_wrong = gram_schmidt([v1, v2, v3], correct_order=False)
print("\nGram matrix E^H E (wrong coefficient order):\n", np.round(E_wrong.conj().T @ E_wrong, 4) + (0 + 0j))

### 4.6 Comparison with `np.linalg.qr`

QR factorisation is Gram–Schmidt done stably. Its $Q$ spans the same nested subspaces, and each column can differ from ours only by a unit factor. (LAPACK keeps the diagonal of $R$ real, so here the factors are $\pm 1$.) More generally, in $\mathbb{C}^n$ a unit vector is only determined up to a **phase** $e^{i\theta}$, not just up to sign: multiplying any column of an orthonormal basis by $e^{i\theta}$ gives another orthonormal basis of the same subspaces.

In [ ]:
import numpy as np


def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w = v.astype(complex)
        for e in orthonormal:
            w = w - np.vdot(e, v) * e
        orthonormal.append(w / np.linalg.norm(w))
    return np.column_stack(orthonormal)


V = np.column_stack([[1, 1j, 0], [1, 1, 1j], [0, 1j, 1 + 1j]])
E = gram_schmidt(list(V.T))
Q, R = np.linalg.qr(V)

phases = np.array([np.vdot(Q[:, k], E[:, k]) for k in range(3)])
print("<q_k, e_k> for each column:", np.round(phases, 6))
print("all of modulus 1:", np.allclose(np.abs(phases), 1))
print("E = Q diag(phases):", np.allclose(E, Q * phases))

rephased = E * np.exp(1j * np.array([0.7, -2.0, 1.3]))
print("columns times e^{i theta} are still orthonormal:", np.allclose(rephased.conj().T @ rephased, np.eye(3)))

## 5. Visualizations

### 5.1 The inner product in $\mathbb{C}^1$: an angle and a product of lengths

For single numbers, $\langle w, z\rangle = \bar w z$ has modulus $|w||z|$ and argument "angle from $w$ to $z$". Its real part is the ordinary dot product of the two arrows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

w = 1.5 * np.exp(0.4j)
z = 1.2 * np.exp(1.5j)
inner = np.conj(w) * z

fig, ax = plt.subplots(figsize=(7, 7))
for number, label, color in [(w, "$w$", "tab:blue"), (z, "$z$", "tab:orange"),
                             (inner, "$\\langle w, z\\rangle = \\bar{w}z$", "crimson")]:
    ax.annotate("", xy=(number.real, number.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, linewidth=2))
    ax.text(number.real * 1.08, number.imag * 1.08 + 0.05, label, color=color, fontsize=12)

arc_w = np.linspace(np.angle(w), np.angle(z), 50)
ax.plot(0.6 * np.cos(arc_w), 0.6 * np.sin(arc_w), color="gray")
arc_inner = np.linspace(0, np.angle(inner), 50)
ax.plot(0.45 * np.cos(arc_inner), 0.45 * np.sin(arc_inner), color="crimson", linestyle=":")
ax.plot([inner.real, inner.real], [0, inner.imag], "--", color="crimson", linewidth=1)
ax.text(inner.real + 0.03, -0.18, "Re = dot product of the arrows", color="crimson", fontsize=9)

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-0.8, 2.2)
ax.set_ylim(-0.4, 2.2)
ax.set_aspect("equal")
ax.set_title("$\\bar{w}z$: modulus $|w||z|$, argument $\\arg z - \\arg w$")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
plt.show()
print(f"|<w,z>| = {abs(inner):.4f} = |w||z| = {abs(w) * abs(z):.4f};  arg = {np.angle(inner):.4f} = 1.5 - 0.4")

### 5.2 Why conjugate: $z\cdot z$ wanders, $\langle z, z\rangle$ does not

Take $z = (1, e^{i\varphi})$, a vector whose length clearly does not depend on $\varphi$. The unconjugated $z \cdot z = 1 + e^{2i\varphi}$ runs around a circle through $0$ in the complex plane; the Hermitian $\langle z, z\rangle = 2$ stays put on the positive real axis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

phi = np.linspace(0, np.pi, 200)
plain = 1 + np.exp(2j * phi)
hermitian = np.full_like(phi, 2.0)

fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(plain.real, plain.imag, c=phi, cmap="viridis", s=12, label="$z \\cdot z = 1 + e^{2i\\varphi}$")
ax.plot(hermitian, np.zeros_like(phi), "o", color="crimson", markersize=10, label="$\\langle z, z\\rangle = 2$ for every $\\varphi$")
ax.plot(0, 0, "kx", markersize=12)
ax.annotate("$z\\cdot z = 0$ at $\\varphi = \\pi/2$", (0, 0), textcoords="offset points", xytext=(10, -20))
fig.colorbar(scatter, ax=ax, label="$\\varphi$", shrink=0.8)

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.set_title("$z = (1, e^{i\\varphi})$: plain square vs Hermitian norm")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.show()

### 5.3 Cauchy–Schwarz: normalised inner products fill the unit disk

For random pairs in $\mathbb{C}^3$, plot $\frac{\langle u, v\rangle}{\lVert u\rVert\lVert v\rVert}$ as a point in the complex plane. Every point lies in the closed unit disk; the boundary is reached only by dependent pairs (red), which land exactly at the phase of the scalar relating them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(5)
normalised = []
for _ in range(3000):
    u = rng.normal(size=3) + 1j * rng.normal(size=3)
    v = rng.normal(size=3) + 1j * rng.normal(size=3)
    normalised.append(np.vdot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))
normalised = np.array(normalised)

u = rng.normal(size=3) + 1j * rng.normal(size=3)
scalars = np.exp(1j * np.linspace(0, 2 * np.pi, 12, endpoint=False)) * 1.7
dependent = np.array([np.vdot(u, c * u) / (np.linalg.norm(u) * np.linalg.norm(c * u)) for c in scalars])

circle = np.exp(1j * np.linspace(0, 2 * np.pi, 300))
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(normalised.real, normalised.imag, s=4, alpha=0.4, label="random pairs")
ax.plot(dependent.real, dependent.imag, "o", color="crimson", label="$v = c\\,u$ (dependent)")
ax.plot(circle.real, circle.imag, color="black", linewidth=1)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect("equal")
ax.set_title("$\\langle u, v\\rangle / (\\|u\\|\\,\\|v\\|)$ always lies in the unit disk")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.show()
print("max modulus of random pairs:", np.abs(normalised).max().round(4), " dependent pairs:", np.abs(dependent).round(12).min())

### 5.4 What an orthonormal pair in $\mathbb{C}^3$ looks like

A complex vector is a list of arrows, one Argand diagram per component. Below are the orthonormal pair

$$
e_1 = \tfrac{1}{\sqrt3}(1, 1, 1), \qquad e_2 = \tfrac{1}{\sqrt3}(1, \omega, \omega^2), \qquad \omega = e^{2\pi i/3},
$$

(two columns of the DFT matrix of notebook 05). Orthogonality is not visible arrow by arrow; it is the statement that the three products $\overline{e_{1j}}\,e_{2j}$ (green) **add up to zero** — placed head to tail in the last panel, they close into a triangle.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

omega = np.exp(2j * np.pi / 3)
e1 = np.array([1, 1, 1]) / np.sqrt(3)
e2 = np.array([1, omega, omega**2]) / np.sqrt(3)
products = np.conj(e1) * e2
print("<e1, e2> =", np.round(np.vdot(e1, e2), 12) + 0, "  norms:", np.linalg.norm(e1), np.linalg.norm(e2))

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for j in range(3):
    ax = axes[j]
    # decreasing widths so overlapping arrows (component 1) stay visible
    for number, label, color, width in [(e1[j], "$e_1$", "tab:blue", 5), (e2[j], "$e_2$", "tab:orange", 2.5),
                                        (products[j], "$\\bar{e}_1 e_2$", "tab:green", 2)]:
        ax.annotate("", xy=(number.real, number.imag), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="->", color=color, linewidth=width))
        ax.plot([], [], color=color, label=label)
    ax.set_title(f"component {j + 1}")

ax = axes[3]
position = 0
for product in products:
    ax.annotate("", xy=((position + product).real, (position + product).imag), xytext=(position.real, position.imag),
                arrowprops=dict(arrowstyle="->", color="tab:green", linewidth=2))
    position = position + product
ax.plot(0, 0, "ko")
ax.set_title("products head to tail: sum $= 0$")

for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.7, 0.7)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. For $u = (2i, 1)$ and $v = (1 - i, 3)$ compute $\langle u, v\rangle$, $\langle v, u\rangle$, $\lVert u\rVert$ and $\lVert v\rVert$.
2. Find $\lVert(3 + 4i,\ 1 - i,\ 2i)\rVert$, and the unit vector in its direction.
3. Among $a = (1, i)$, $b = (1, -i)$, $c = (i, 1)$, which pairs are orthogonal? Which pair spans the same complex line?

### Intermediate

4. **Draw it first, then compute.** Work in $\mathbb{C}^1$ with $w = 1 + i$. Sketch a $z$ with $\langle w, z\rangle$ positive real, one with $\langle w, z\rangle$ purely imaginary, and decide whether any nonzero $z$ has $\langle w, z\rangle = 0$. Then compute $\langle w, w\rangle$ and $\langle w, iw\rangle$.
5. Apply Gram–Schmidt by hand to $v_1 = (1, i)$, $v_2 = (2, 1 + i)$ in $\mathbb{C}^2$.
6. Show from the definition that $\langle u, cv\rangle = c\langle u, v\rangle$ but $\langle cu, v\rangle = \bar c\langle u, v\rangle$, and that $\lVert cv\rVert = |c|\lVert v\rVert$. Which of these changes in the "conjugate the second argument" convention?

### Advanced

7. Prove that equality $|\langle u, v\rangle| = \lVert u\rVert\lVert v\rVert$ holds if and only if $u$ and $v$ are linearly dependent over $\mathbb{C}$.
8. Prove the **polarization identity** $\langle u, v\rangle = \frac14\sum_{k=0}^{3} i^{-k}\,\lVert u + i^k v\rVert^2$: the norm alone determines the inner product.
9. Show that $\langle u, v\rangle = 0$ if and only if $u$ is perpendicular to both $v$ and $iv$ in $\mathbb{R}^{2n}$. Deduce that in $\mathbb{C}^1$ no two nonzero numbers are orthogonal.

## 7. Solutions

1. $\langle u, v\rangle = \overline{2i}(1 - i) + 1\cdot 3 = -2i + 2i^2 + 3 = 1 - 2i$; $\langle v, u\rangle = 1 + 2i$; $\lVert u\rVert = \sqrt{4 + 1} = \sqrt5$; $\lVert v\rVert = \sqrt{2 + 9} = \sqrt{11}$.
2. $25 + 2 + 4 = 31$, so the norm is $\sqrt{31}$ and the unit vector is $\frac{1}{\sqrt{31}}(3 + 4i,\ 1 - i,\ 2i)$.
3. $\langle a, b\rangle = 0$ and $\langle a, c\rangle = 0$; $\langle b, c\rangle = 2i \ne 0$. Since $c = ib$, the pair $b, c$ spans one complex line, and $a$ is orthogonal to that whole line.
4. $\langle w, z\rangle = \bar w z$ has argument $\arg z - \arg w$. Positive real: $z$ points the same way as $w$ (e.g. $z = w$, giving $|w|^2 = 2$). Purely imaginary: $z$ is $w$ turned by $\pm 90°$ (e.g. $z = iw = -1 + i$, giving $\langle w, iw\rangle = i|w|^2 = 2i$). Zero: impossible for $z \ne 0$, since $|\bar w z| = |w||z| > 0$. So $iw$ is perpendicular to $w$ as an arrow but not orthogonal in $\mathbb{C}^1$.
5. $e_1 = \frac{1}{\sqrt2}(1, i)$. $\langle e_1, v_2\rangle = \frac{1}{\sqrt2}\big(2 + (-i)(1 + i)\big) = \frac{3 - i}{\sqrt2}$. $w_2 = (2, 1 + i) - \frac{3 - i}{2}(1, i) = \left(\frac{1 + i}{2},\ \frac{1 - i}{2}\right)$, with $\lVert w_2\rVert^2 = \frac12 + \frac12 = 1$, so $e_2 = w_2$. Check: $\langle e_1, e_2\rangle = \frac{1}{\sqrt2}\cdot\frac{(1 + i) - i(1 - i)}{2} = \frac{1}{\sqrt2}\cdot\frac{1 + i - i - 1}{2} = 0$. ✓
6. $\langle u, cv\rangle = \sum\bar u_j(cv_j) = c\sum\bar u_jv_j$; $\langle cu, v\rangle = \sum\overline{cu_j}v_j = \bar c\sum\bar u_jv_j$; $\lVert cv\rVert^2 = \langle cv, cv\rangle = \bar c c\lVert v\rVert^2 = |c|^2\lVert v\rVert^2$. In the other convention the roles of the slots swap (scalars come out of the first slot unchanged and out of the second with a bar); the norm is the same.
7. If $v = 0$ both sides are $0$. Otherwise, from §2.6, $\lVert u\rVert^2 - \frac{|\langle v, u\rangle|^2}{\lVert v\rVert^2} = \lVert u - \operatorname{proj}_v u\rVert^2$, which is zero iff $u = \operatorname{proj}_v u = cv$ for $c = \langle v, u\rangle/\lVert v\rVert^2$. Conversely if $u = cv$ then $|\langle u, v\rangle| = |\bar c|\lVert v\rVert^2 = \lVert u\rVert\lVert v\rVert$.
8. With $c = i^k$ (so $|c| = 1$): $\lVert u + cv\rVert^2 = \lVert u\rVert^2 + \lVert v\rVert^2 + cX + \bar c\bar X$ where $X = \langle u, v\rangle$. Multiply by $\bar c = i^{-k}$ and sum over $k = 0..3$: the constant term is multiplied by $\sum_k i^{-k} = 0$, the $cX$ term gives $\sum_k \bar c c X = 4X$, and the last gives $\bar X\sum_k i^{-2k} = \bar X\sum_k(-1)^k = 0$. Hence the sum is $4X$.
9. By §2.4, $\operatorname{Re}\langle u, v\rangle$ is the real dot product of the realified vectors, and $\operatorname{Im}\langle u, v\rangle = \operatorname{Re}\big(-i\langle u, v\rangle\big) = -\operatorname{Re}\langle u, iv\rangle$. So $\langle u, v\rangle = 0$ iff $u \cdot_{\mathbb{R}} v = 0$ and $u \cdot_{\mathbb{R}} (iv) = 0$. In $\mathbb{C}^1 \cong \mathbb{R}^2$, $v$ and $iv$ already span the whole plane for $v \ne 0$, so only $u = 0$ is perpendicular to both.

Numerical checks:

In [ ]:
import numpy as np

u, v = np.array([2j, 1]), np.array([1 - 1j, 3])
print("Ex 1:", np.vdot(u, v), np.vdot(v, u), np.vdot(u, u).real, np.vdot(v, v).real)
x = np.array([3 + 4j, 1 - 1j, 2j])
print("Ex 2: ||x||^2 =", np.vdot(x, x).real, " unit vector norm:", np.linalg.norm(x / np.linalg.norm(x)))
a, b, c = np.array([1, 1j]), np.array([1, -1j]), np.array([1j, 1])
print("Ex 3:", np.vdot(a, b), np.vdot(a, c), np.vdot(b, c), " c = i b:", np.allclose(c, 1j * b))
w = 1 + 1j
print("Ex 4: <w,w> =", np.conj(w) * w, " <w,iw> =", np.conj(w) * (1j * w))
e1 = np.array([1, 1j]) / np.sqrt(2)
w2 = np.array([2, 1 + 1j]) - np.vdot(e1, np.array([2, 1 + 1j])) * e1
print("Ex 5: w2 =", np.round(w2, 12), " ||w2|| =", np.linalg.norm(w2), " <e1,w2> ~ 0:", abs(np.vdot(e1, w2)) < 1e-12)

rng = np.random.default_rng(7)
u = rng.normal(size=3) + 1j * rng.normal(size=3)
v = rng.normal(size=3) + 1j * rng.normal(size=3)
polarised = sum(1j ** (-k) * np.linalg.norm(u + 1j**k * v) ** 2 for k in range(4)) / 4
print("Ex 8: polarization matches <u,v>:", np.isclose(polarised, np.vdot(u, v)))

## 8. Conclusion

- In $\mathbb{C}^n$ the plain dot product fails ($(1, i)\cdot(1, i) = 0$); the **Hermitian inner product** $\langle u, v\rangle = \sum\overline{u_j}v_j = u^Hv$ fixes it, just as $|z|^2 = \bar z z$ does in one dimension.
- It is conjugate-symmetric, linear in the second slot and conjugate-linear in the first (numpy's `np.vdot` convention); some math texts swap the slots. `np.dot`, `@` and `np.inner` never conjugate.
- $\lVert v\rVert^2 = \sum|v_j|^2$ equals the Euclidean norm in $\mathbb{R}^{2n}$, and $\operatorname{Re}\langle u, v\rangle$ is the real dot product; in $\mathbb{C}^1$, $\bar wz$ encodes the angle from $w$ to $z$.
- Complex orthogonality ($\langle u, v\rangle = 0$) means perpendicular to both $v$ and $iv$: orthogonality of complex lines.
- Cauchy–Schwarz $|\langle u, v\rangle| \le \lVert u\rVert\lVert v\rVert$ holds with the projection proof; normalised inner products lie in the unit disk.
- Gram–Schmidt works verbatim provided the coefficient is $\langle e_j, v\rangle$ (basis vector in the conjugated slot); unit vectors are determined only up to a phase $e^{i\theta}$.

**Next:** notebook 04 studies the matrices that interact best with this inner product — Hermitian matrices ($A^H = A$, real eigenvalues) and unitary matrices ($U^HU = I$, which preserve $\langle\cdot,\cdot\rangle$).